In [1]:
import rasterio
import numpy as np
import torch

ModuleNotFoundError: No module named 'rasterio'

In [ ]:
import os

# Folder where bands are stored
band_folder = "Browser_images"

# List the bands in the correct order (adjust based on LCZNet requirements)
band_order = ["B02.tiff", "B03.tiff", "B04.tiff", "B05.tiff",
              "B06.tiff", "B07.tiff", "B8A.tiff", "B08.tiff",
              "B09.tiff", "B11.tiff", "B12.tiff"]

# Create full file paths
band_paths = [os.path.join(band_folder, band) for band in band_order]


In [ ]:
# Initialize an empty list to hold the band data
bands = []

for band_path in band_paths:
    with rasterio.open(band_path) as src:
        bands.append(src.read(1))  # Read the first band of each file

# Stack bands into a single numpy array [channels, height, width]
stacked_bands = np.stack(bands, axis=0).astype(np.float32)

# Normalize the data (assuming LCZNet expects normalized input)
stacked_bands /= 10000.0  # Adjust this normalization based on your data


In [ ]:
# Convert numpy array to PyTorch tensor and add a batch dimension
input_tensor = torch.from_numpy(stacked_bands).unsqueeze(0)  # Shape: [1, channels, height, width]



In [ ]:
# Load LCZNet model
model = torch.load("LCZNet_checkpoint.pth")
model.eval()

# Inference
with torch.no_grad():
    output = model(input_tensor)

# Get predicted LCZ classes
lcz_map = torch.argmax(output, dim=1).squeeze().numpy()


In [ ]:
# Use the transform and CRS from one of the input bands (e.g., B02.tiff)
with rasterio.open(band_paths[0]) as src:
    transform = src.transform
    crs = src.crs

# Save the LCZ map as a GeoTIFF
output_file = "lcz_map.tif"
with rasterio.open(
    output_file, "w",
    driver="GTiff",
    height=lcz_map.shape[0],
    width=lcz_map.shape[1],
    count=1,
    dtype=lcz_map.dtype,
    crs=crs,
    transform=transform
) as dst:
    dst.write(lcz_map, 1)

print(f"LCZ map saved at {output_file}")
